# Final experiment: tuning → FINAL TEST → explanations → app model

Run top to bottom. Each section saves its output, so after a restart you can re-run without redoing slow steps.

| Section | Time | Saves |
|---|---|---|
| 1. Data + features + Dixon–Coles | ~1 min | `poisson_probs.csv` |
| 2. Tuning (loads saved results if they exist) | 0 or ~15 min | `tuning_results.csv` |
| 3. **FINAL TEST, once** | ~1 min | `results_table.csv`, `test_predictions.npz`, models |
| 4. Bootstrap (real or luck?) | ~10 s | `rq_bootstrap.csv` |
| 5. Charts + tables | ~1 min | `results/figures/*.png` |
| 6. App model | ~1 min | `results/models/app_model.joblib` |

**Rule:** never change settings after looking at Section 3. Re-running with the same settings is fine (same numbers).

In [7]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("working in:", os.getcwd())

working in: c:\Users\PCN\Projects\FootballPrediction


## 1. Data, features, Dixon–Coles probabilities

In [8]:
from pathlib import Path
import numpy as np, pandas as pd
from src.load_data import read_matches
from src.features import build_features, SETUPS
from src.poisson import walk_forward_probs, fill_missing_probs

m = read_matches()
f = build_features(m)

probs_path = Path("data/processed/poisson_probs.csv")
probs = pd.read_csv(probs_path) if probs_path.exists() else None
if probs is None or "pois_rho" not in probs or probs["pois_rho"].abs().sum() == 0:
    print("computing Dixon-Coles probabilities (~1 min)...")
    probs = walk_forward_probs(m, xi=0.0019, dixon_coles=True)
    probs.to_csv(probs_path, index=False)
f = f.merge(fill_missing_probs(probs), on="match_id", how="left")
print(len(f), "matches | avg rho:", round(probs.pois_rho.mean(), 3))

7600 matches | avg rho: -0.044


In [9]:
import numpy as np, pandas as pd
sample = f[f.season >= "1011"].sample(3, random_state=7)

def side_stats(p, team):
    """goals for, goals against, shots on target for, against - from the team's point of view"""
    if p.home == team:
        return p.hg, p.ag, p.hst, p.ast
    return p.ag, p.hg, p.ast, p.hst

for _, r in sample.iterrows():
    print(f"\n=== {r.date:%d %b %Y}: {r.home} vs {r.away} (result {r.result}) ===")
    for side, team in [("home", r.home), ("away", r.away)]:
        past = m[((m.home == team) | (m.away == team)) & (m.date < r.date)].sort_values("date")
        last5 = past.tail(5)
        pts, gf, ga, sf, sa = [], [], [], [], []
        for _, p in last5.iterrows():
            g_for, g_ag, s_for, s_ag = side_stats(p, team)
            pts.append(3 if g_for > g_ag else 1 if g_for == g_ag else 0)
            gf.append(g_for); ga.append(g_ag); sf.append(s_for); sa.append(s_ag)
        rest = min((r.date - past.date.iloc[-1]).days, 14)
        print(f"{side.upper()}: {team}")
        for _, p in last5.iterrows():
            print(f"    {p.date:%d/%m/%Y}  {p.home} {p.hg}-{p.ag} {p.away}")
        print(f"  form (pts/game)   hand {np.mean(pts):.3f}  | pipeline {r[f'{side}_form_pts5']:.3f}")
        print(f"  goals for/game    hand {np.mean(gf):.3f}  | pipeline {r[f'{side}_form_gf5']:.3f}")
        print(f"  goals against     hand {np.mean(ga):.3f}  | pipeline {r[f'{side}_form_ga5']:.3f}")
        print(f"  shots-on-target   hand {sum(sf) / (sum(sf) + sum(sa)):.3f}  | pipeline {r[f'{side}_sot_ratio5']:.3f}")
        print(f"  rest days         hand {rest}      | pipeline {r[f'{side}_rest_days']:.0f}")

    meet = m[(((m.home == r.home) & (m.away == r.away)) | ((m.home == r.away) & (m.away == r.home)))
             & (m.date < r.date)].sort_values("date").tail(6)
    w = int(((meet.home == r.home) & (meet.result == "H")).sum() + ((meet.away == r.home) & (meet.result == "A")).sum())
    d = int((meet.result == "D").sum()); n = len(meet)
    print(f"  head-to-head: last {n} meetings = {w}W {d}D {n - w - d}L for {r.home}")
    print(f"    hand ({w} + 0.5x{d} + 1.5) / ({n} + 3) = {(w + 0.5 * d + 1.5) / (n + 3):.3f}  | pipeline {r.h2h_score:.3f}")

    exp_h = 1 / (1 + 10 ** (-(r.elo_home + 60 - r.elo_away) / 400))
    act_h = {"H": 1.0, "D": 0.5, "A": 0.0}[r.result]
    nxt = f[((f.home == r.home) | (f.away == r.home)) & (f.date > r.date) & (f.season == r.season)].sort_values("date").head(1)
    print(f"  Elo before: {r.home} {r.elo_home:.1f}, {r.away} {r.elo_away:.1f} -> expected home score {exp_h:.3f}")
    if len(nxt):
        nx = nxt.iloc[0]
        pipe_next = nx.elo_home if nx.home == r.home else nx.elo_away
        print(f"    {r.home} after: hand {r.elo_home:.1f} + 20x({act_h} - {exp_h:.3f}) = {r.elo_home + 20 * (act_h - exp_h):.1f}  | pipeline (next match) {pipe_next:.1f}")


=== 29 Feb 2020: Watford vs Liverpool (result H) ===
HOME: Watford
    18/01/2020  Watford 0-0 Tottenham
    21/01/2020  Aston Villa 2-1 Watford
    01/02/2020  Watford 2-3 Everton
    08/02/2020  Brighton 1-1 Watford
    23/02/2020  Man United 3-0 Watford
  form (pts/game)   hand 0.400  | pipeline 0.400
  goals for/game    hand 0.800  | pipeline 0.800
  goals against     hand 1.800  | pipeline 1.800
  shots-on-target   hand 0.278  | pipeline 0.278
  rest days         hand 6      | pipeline 6
AWAY: Liverpool
    23/01/2020  Wolves 1-2 Liverpool
    29/01/2020  West Ham 0-2 Liverpool
    01/02/2020  Liverpool 4-0 Southampton
    15/02/2020  Norwich 0-1 Liverpool
    24/02/2020  Liverpool 3-2 West Ham
  form (pts/game)   hand 3.000  | pipeline 3.000
  goals for/game    hand 2.400  | pipeline 2.400
  goals against     hand 0.600  | pipeline 0.600
  shots-on-target   hand 0.673  | pipeline 0.673
  rest days         hand 5      | pipeline 5
  head-to-head: last 6 meetings = 0W 1D 5L for Wa

## 2. Tuning on 5 seasons (2018/19–2022/23)
Loads `results/tuning_results.csv` if it exists. The second cell tries the extra (simpler) settings and keeps any that are better; it's safe to run again.

In [10]:
from src.models import tune_rolling, load_tuned, extend_tuning
tune_path = Path("results/tuning_results.csv")
if tune_path.exists():
    tuned = load_tuned(tune_path)
    print("loaded saved tuning results")
else:
    print("tuning (~15 min)...")
    tuned = tune_rolling(f, SETUPS)
    tuned.to_csv(tune_path, index=False)
tuned.pivot(index="model", columns="setup", values="rps").round(4)

loaded saved tuning results


setup,A,B,C,D
model,,,,
LR,0.2100,0.2027,0.2025,0.2015
RF,0.2107,0.2042,0.2043,0.2027
XGB,0.2112,0.2037,0.2037,0.2031


In [11]:
tuned = extend_tuning(f, tuned, SETUPS)
tuned.to_csv(tune_path, index=False)
print("FROZEN SETTINGS (copy these into the Results Log):")
tuned[["model", "setup", "params", "accuracy", "rps"]]

FROZEN SETTINGS (copy these into the Results Log):


,model,setup,params,accuracy,rps
0,LR,A,{'C': 0.01},0.5268,0.2100
1,RF,A,"{'max_depth': 4, 'min_samples_leaf': 50}",0.5295,0.2107
2,XGB,A,"{'max_depth': 1, 'learning_rate': 0.01, 'n_est...",0.5247,0.2112
3,LR,B,{'C': 0.01},0.5405,0.2027
4,RF,B,"{'max_depth': None, 'min_samples_leaf': 20}",0.5347,0.2042
5,XGB,B,"{'max_depth': 1, 'learning_rate': 0.01, 'n_est...",0.5395,0.2037
6,LR,C,{'C': 0.01},0.5416,0.2025
7,RF,C,"{'max_depth': 6, 'min_samples_leaf': 20}",0.5384,0.2043
8,XGB,C,"{'max_depth': 1, 'learning_rate': 0.01, 'n_est...",0.5400,0.2037
9,LR,D,{'C': 0.003},0.5447,0.2015


## 3. ⚠️ FINAL TEST: 2023/24 + 2024/25, run once
Trains every tuned model on 2010/11–2022/23 and scores it on the 760 test matches. It refuses to run a second time. If you need to reproduce it after a crash (same settings), use `force=True`.

In [15]:
from src.models import run_final_test
final = run_final_test(f, tuned, SETUPS, force=True)
final.sort_values("rps")[["model", "setup", "accuracy", "macro_F1", "brier", "rps"]]

,model,setup,accuracy,macro_F1,brier,rps
13,Bookmaker (Bet365),-,0.5684,0.4222,0.5559,0.1906
3,LR,B,0.5645,0.4168,0.5677,0.1951
9,LR,D,0.5605,0.4156,0.5672,0.1951
6,LR,C,0.5632,0.4172,0.5680,0.1953
11,XGB,D,0.5592,0.4132,0.5719,0.1969
10,RF,D,0.5474,0.4072,0.5716,0.1972
5,XGB,B,0.5618,0.4154,0.5734,0.1974
8,XGB,C,0.5618,0.4154,0.5734,0.1974
4,RF,B,0.5579,0.4147,0.5727,0.1976
12,Poisson (Dixon-Coles),-,0.5474,0.4071,0.5717,0.1979


## 4. Are the differences real or luck? (bootstrap, 1,000 resamples)
`diff` < 0 means the newer setup is better. `clear = True` means the whole 95% range is on one side of zero.

In [13]:
from src import analysis as an
final = pd.read_csv("results/results_table.csv")
y, match_id, preds = an.load_predictions()
rq = an.research_question_table(y, preds)
rq.to_csv("results/rq_bootstrap.csv", index=False)
rq

,question,model,compare,diff,ci_low,ci_high,p_better,clear
0,RQ1 features,LR,B vs A,-0.0097,-0.0141,-0.0050,1.000,True
1,RQ2 head-to-head,LR,C vs B,0.0002,-0.0004,0.0008,0.243,False
2,RQ3 hybrid,LR,D vs C,-0.0002,-0.0014,0.0010,0.645,False
3,RQ1 features,RF,B vs A,-0.0072,-0.0109,-0.0034,1.000,True
4,RQ2 head-to-head,RF,C vs B,0.0003,-0.0008,0.0014,0.282,False
5,RQ3 hybrid,RF,D vs C,-0.0008,-0.0024,0.0008,0.828,False
6,RQ1 features,XGB,B vs A,-0.0073,-0.0112,-0.0032,1.000,True
7,RQ2 head-to-head,XGB,C vs B,0.0000,-0.0000,0.0000,0.501,False
8,RQ3 hybrid,XGB,D vs C,-0.0005,-0.0017,0.0006,0.787,False


## 5. Charts and tables for the report

In [16]:
print(an.plot_setup_comparison(final))

ml = final[final.model.isin(["LR", "RF", "XGB"])].sort_values("rps")
best = ml.iloc[0]
best_key = f"{best.model}_{best.setup}"
print("best model:", best_key, "| rps", best.rps, "| accuracy", best.accuracy)

test = f.set_index("match_id").loc[match_id].reset_index()
print(an.plot_confusion(y, preds[best_key], f"Confusion matrix: {an.MODEL_NAMES[best.model]}, setup {best.setup}")[0])
draws, draw_summary = an.draw_analysis(y, preds[best_key], test.elo_diff)
print(an.plot_draws(draws))
print(draw_summary)
draws

results\figures\setup_comparison.png
best model: LR_B | rps 0.1951 | accuracy 0.5645
results\figures\confusion_matrix.png
results\figures\draw_analysis.png
{'draw_rate': 0.23, 'draw_picked': 0.001, 'max_draw_prob': 0.411}


,elo_gap,matches,actual_draw_rate,avg_predicted_draw,picked_draw_share
0,0-50,224,0.246,0.263,0.004
1,50-100,205,0.229,0.251,0.000
2,100-200,240,0.275,0.221,0.000
3,200+,91,0.077,0.175,0.000


In [17]:
print(an.plot_calibration(y, {
    f"{an.MODEL_NAMES[best.model]} ({best.setup})": (preds[best_key], an.COLORS[best.model]),
    "Bookmaker": (preds["Bookmaker (Bet365)"], an.GREY),
}))

import joblib
lr = joblib.load("results/models/LR_D.joblib")
coefs = an.lr_coefficients(lr, SETUPS["D"])
print(an.plot_lr_coefficients(coefs))
coefs.head(12).round(3)

results\figures\calibration_home.png
results\figures\lr_coefficients.png


,home,draw,away,home_minus_away,importance
elo_home,0.127,-0.033,-0.094,0.221,0.127
elo_diff,0.118,-0.014,-0.104,0.223,0.118
pois_pA,-0.068,-0.042,0.110,-0.178,0.110
pois_pH,0.073,0.021,-0.093,0.166,0.093
diff_sot_ratio5,0.071,0.007,-0.078,0.149,0.078
pois_pD,-0.041,0.061,-0.020,-0.021,0.061
away_sot_ratio5,-0.059,-0.000,0.059,-0.119,0.059
elo_away,-0.045,-0.013,0.057,-0.102,0.057
home_sot_ratio5,0.040,0.010,-0.050,0.091,0.050
home_ga_season,-0.016,0.037,-0.021,0.005,0.037


In [18]:
xgb = joblib.load("results/models/XGB_C.joblib")
imp, path, rank = an.shap_importance(xgb, test[SETUPS["C"]])
print(path)
print("head-to-head rank (out of", len(rank), "):", rank.get("h2h_score"), "| h2h_n:", rank.get("h2h_n"))
imp.head(12).round(4)

results\figures\shap_xgb.png
head-to-head rank (out of 32 ): 31 | h2h_n: 32


elo_diff           0.2731
diff_sot_ratio5    0.1056
elo_home           0.0303
away_gf_season     0.0294
diff_gf_season     0.0207
away_sot_ratio5    0.0088
home_gf_season     0.0062
diff_ga_season     0.0028
home_venue_pts5    0.0027
away_ppg_season    0.0019
diff_ppg_season    0.0013
away_ga_season     0.0007
dtype: float32

In [19]:
path, ha = an.plot_home_advantage(m)
print(path)

from src.predict import poisson_for_fixture
home, away = "Arsenal", "Chelsea"
(pH, pD, pA), grid, xg, rho = poisson_for_fixture(m, home, away, m.date.max() + pd.Timedelta(days=7))
print(an.plot_scoreline_grid(grid, home, away))
print(f"{home} {pH:.0%} / draw {pD:.0%} / {away} {pA:.0%} | xG {xg[0]:.2f}-{xg[1]:.2f} | rho {rho:.3f}")
ha.tail(8)

results\figures\home_advantage.png
results\figures\scoreline_example.png
Arsenal 57% / draw 24% / Chelsea 20% | xG 1.78-0.94 | rho -0.017


result,H,D,A
season,,,
1718,0.455,0.261,0.284
1819,0.476,0.187,0.337
1920,0.453,0.242,0.305
2021,0.379,0.218,0.403
2122,0.429,0.232,0.339
2223,0.484,0.229,0.287
2324,0.461,0.216,0.324
2425,0.408,0.245,0.347


## 6. App model (trained on all seasons 2010/11–2024/25)
Uses the tuned LR setup D settings. If your final results pick a different best model, change `model_name` / `setup`.

In [ ]:
from src.predict import train_app_model, predict_fixture
row = tuned[(tuned.model == "LR") & (tuned.setup == "D")].iloc[0]
train_app_model("LR", "D", row.params)
r = predict_fixture("Arsenal", "Chelsea", m)
print(r["source"], [round(p, 3) for p in r["probs"]], r["top_scores"][:3])

saved LR setup D -> results\models\app_model.joblib
LR (setup D) [0.599, 0.23, 0.171] [(np.float64(0.11466712394480548), '1-0'), (np.float64(0.1119363506382625), '1-1'), (np.float64(0.10393912497929074), '2-0')]


: 

## Done
Push everything:
```
git add results data notebooks
git commit -m "Final test, figures, app model"
git push
```
Then run the app: `streamlit run app/app.py`.